In [79]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import optimize
from Utils import *
import matplotlib.pyplot as plt
import os
from tqdm import tqdm
%load_ext autoreload 
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [80]:
#utils conversions and constants

deg = np.pi/180
AU_m = 1.496e11 #m
M_sun = 1.9891e30
G = 6.67430e-11 # m^3 / (kg s^2)
year = 365.25*24*3600 #s

G_cu = G * (1/AU_m)**3 * (M_sun) * (year)**2  
mu = G_cu

print(f"G in Canonical units: {G_cu} AU^3/ (M_sun year^2)")
print(f"mu in Canonical units: {mu} AU^3/year^2")

G in Canonical units: 39.48894168123623 AU^3/ (M_sun year^2)
mu in Canonical units: 39.48894168123623 AU^3/year^2


In [81]:
N = int(1e7)

a_coplanar = np.random.uniform(0, 2, N)
e_coplanar = np.zeros_like(a_coplanar) + 0.01
i_coplanar = np.zeros_like(a_coplanar) + 0.01
w_coplanar = np.zeros_like(a_coplanar) + 0.01
Omega_coplanar = np.zeros_like(a_coplanar) + 0.01
M_coplanar = np.random.uniform(0, 2, N)

In [82]:
state_vector_coplanar = np.zeros((N, 6))
for element in tqdm(range(N)):
    a = a_coplanar[element]
    e = e_coplanar[element] 
    i = i_coplanar[element]
    Omega = Omega_coplanar[element]
    w = w_coplanar[element]
    M = M_coplanar[element]
    q=a*(1-e)

    elements_spice = np.array([q, e, i, Omega, w, M, 0.0, mu])
    et = 0 
    state_vector_spice = spy.conics(elements_spice, et)
    state_vector_coplanar[element] = state_vector_spice

100%|██████████| 10000000/10000000 [01:32<00:00, 108164.78it/s]


In [96]:
state_vector_coplanar

array([[ 1.17430333e+00,  1.30756159e+00,  1.29579657e-02,
        -3.51568515e+00,  3.20345264e+00,  3.23855669e-02],
       [ 3.72554818e-01,  1.08063797e-01,  1.04336385e-03,
        -2.79938317e+00,  9.74352977e+00,  9.77136167e-02],
       [ 1.01399205e+00,  9.91660962e-01,  9.81504344e-03,
        -3.67707517e+00,  3.81098905e+00,  3.84769689e-02],
       ...,
       [ 1.15529637e+00,  1.56813517e-01,  1.45257747e-03,
        -7.80102067e-01,  5.79609882e+00,  5.80380337e-02],
       [ 4.08675085e-01,  8.59662414e-01,  8.55561267e-03,
        -5.80536106e+00,  2.82316924e+00,  2.88117676e-02],
       [ 4.33148427e-01,  8.30456514e-01,  8.26111117e-03,
        -5.74455942e+00,  3.06001900e+00,  3.11741456e-02]],
      shape=(10000000, 6))

In [97]:
xs = state_vector_coplanar[:,0]
ys = state_vector_coplanar[:,1]
zs = state_vector_coplanar[:,2]
vxs = state_vector_coplanar[:,3]
vys = state_vector_coplanar[:,4]
vzs = state_vector_coplanar[:,5]

In [98]:
np.mean(xs), np.mean(ys), np.mean(zs), np.mean(vxs), np.mean(vys), np.mean(vzs)

(np.float64(0.4243619339850745),
 np.float64(0.718595679440056),
 np.float64(0.007143400128347875),
 np.float64(-6.4152001611569185),
 np.float64(3.900189397841712),
 np.float64(0.03964277465876937))

In [106]:
xs = state_vector_coplanar[:,0]
ys = state_vector_coplanar[:,1]
zs = state_vector_coplanar[:,2]
vxs = state_vector_coplanar[:,3]
vys = state_vector_coplanar[:,4]
vzs = state_vector_coplanar[:,5]

x_cen = 0 #AU
y_cen = 0
z_cen = 0

delta_r = 4  #AU
dx = delta_r
dy = delta_r
dz = delta_r

vx_cen = 0
vy_cen = 0
vz_cen = 0

delta_v = 4  #AU/year
dvx = delta_v
dvy = delta_v
dvz = delta_v

objsx = (abs(xs - x_cen) <= dx) 
objsy = (abs(ys - y_cen) <= dy) 
objsz = (abs(zs - z_cen) <= dz) 
objsvx = (abs(vxs - vx_cen) <= dvx) 
objsvy = (abs(vys - vy_cen) <= dvy) 
objsvz = (abs(vzs - vz_cen) <= dvz) 

objects = objsx * objsy * objsz * objsvx * objsvy * objsvz
print(f'Number of objects inside volume: {objects.sum()}')

Number of objects inside volume: 574698


In [107]:
xsel = xs[objects]
ysel = ys[objects]
zsel = zs[objects]
vxs_sel = vxs[objects]
vys_sel = vys[objects]
vzs_sel = vzs[objects]

X = np.array([xsel[0], ysel[0], zsel[0], vxs_sel[0], vys_sel[0], vzs_sel[0]])
X

array([ 1.17430333,  1.30756159,  0.01295797, -3.51568515,  3.20345264,
        0.03238557])

In [108]:
E = spy.oscelt(X, et=0, mu=mu)
E

array([1.75200132e+00, 1.00000000e-02, 1.00000000e-02, 1.00000000e-02,
       1.00000000e-02, 8.04528831e-01, 0.00000000e+00, 3.94889417e+01])

In [109]:
E = E[:6]
JXoE = calcKeplerianJacobians(mu,E,X)
JXoE

array([[ 6.63561315e-01, -2.72466996e+00,  1.29577498e-04,
        -1.30756159e+00, -1.30762579e+00, -1.31710706e+00],
       [ 7.38861304e-01,  8.22488258e-01, -1.29573178e-02,
         1.17430333e+00,  1.17424332e+00,  1.20013309e+00],
       [ 7.32213266e-03,  8.49721703e-03,  1.29575338e+00,
         0.00000000e+00,  1.18730023e-02,  1.21328438e-02],
       [ 9.93300709e-01, -4.75736563e+00,  3.23850271e-04,
        -3.20345264e+00, -3.20361630e+00, -3.20012319e+00],
       [-9.05084397e-01, -3.98026238e-01, -3.23839476e-02,
        -3.51568515e+00, -3.51551261e+00, -3.56326860e+00],
       [-9.15002485e-03, -3.50445155e-03,  3.23844873e+00,
         0.00000000e+00, -3.48341732e-02, -3.53120744e-02]])

In [110]:
det = np.abs(np.linalg.det(np.linalg.inv(JXoE)))
det

np.float64(60.58629219931834)

In [111]:
a_min = 0 #au
a_max = 2 #au
e_min = 0
e_max = 1
i_min = 0 #rad
i_max = np.pi #rad
Omega_min = 0 #rad
Omega_max = 2*np.pi #rad
w_min = 0 #rad
w_max = 2*np.pi #rad
E_min = 0 #rad
E_max = 2*np.pi #rad

p_E = p_E_uniform(a_min, a_max, e_min, e_max, i_min, i_max, Omega_min, Omega_max, w_min, w_max, E_min, E_max)

print(f'Probability uniform distribution for orbital elements: ', p_E)

Probability uniform distribution for orbital elements:  0.000641623890917771


In [112]:
Delta = (dx) * (dy) * (dz) * (dvx) * (dvy) * (dvz)
n = N * p_E * det * Delta
n

np.float64(1592263169.5240247)